In [1]:
# ============================================================
# Spark Setup
# ============================================================ 
# Create a local Spark session using all available CPU cores. 
# defaultParallelism is printed because it is used later to 
# justify the shuffle partition configuration in Part C. 
# ============================================================

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = (SparkSession.builder
         .appName("capstone")
         .master("local[*]")
         .getOrCreate())


print(spark.version)
print("cores:", spark.sparkContext.defaultParallelism)   
print(spark.sparkContext.uiWebUrl)      

3.4.1
cores: 8
http://localhost:4040


In [2]:
# ============================================================
# Input Data
# ============================================================
# Define the location of the assignment input CSV files.
# ============================================================

BASE = "data"


# ============================================================ 
# Declared Schemas 
# ============================================================ 
# Explicit schemas are used instead of schema inference. 
# The 'ts' column is intentionally read as STRING because the 
# raw data contains invalid timestamp values that must be 
# identified and handled during the cleaning process. 
# ============================================================

orders_schema = """
    order_id INT,
    customer_id INT,
    product_id INT,
    amount DOUBLE,
    ts STRING,
    country STRING
"""

customers_schema = """
    customer_id INT,
    name STRING,
    country STRING,
    signup_date DATE
"""

products_schema = """
    product_id INT,
    category STRING,
    unit_price DOUBLE
"""

# ============================================================ 
# Read Input Data 
# ============================================================

orders = spark.read.csv(
    f"{BASE}/orders.csv",
    header=True,
    schema=orders_schema
)

customers = spark.read.csv(
    f"{BASE}/customers.csv",
    header=True,
    schema=customers_schema
)

products = spark.read.csv(
    f"{BASE}/products.csv",
    header=True,
    schema=products_schema
)

In [3]:
orders.printSchema()
customers.printSchema()
products.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- ts: string (nullable = true)
 |-- country: string (nullable = true)

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- country: string (nullable = true)
 |-- signup_date: date (nullable = true)

root
 |-- product_id: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- unit_price: double (nullable = true)



In [4]:
# ============================================================ 
# Initial Row Counts 
# ============================================================ 
# Record the starting number of rows so that the cleaning 
# process can be fully accounted for later in Part A. 
# ============================================================



print("Initial row counts")
print("------------------")


orders_initial = orders.count()
customers_initial = customers.count()

print(f"Orders: {orders_initial}")
print(f"Customers: {customers_initial}")
print(f"Products: {products.count()}")

Initial row counts
------------------


Orders: 1020025
Customers: 10300
Products: 500


### Cleaning Order and Coustomer

In [5]:
# ============================================================ 
# Part A — Data Cleaning 
# ============================================================

# ------------------------------------------------------------ 
# 1. Remove exact duplicate order rows 
# ------------------------------------------------------------ 
# The assignment specifies exact duplicate rows, meaning all 
# column values are identical. Therefore, dropDuplicates() 
# is applied without specifying individual columns.

orders_dedup  = orders.dropDuplicates()
duplicates_removed = orders_initial - orders_dedup.count()
print(f"Duplicate orders removed: {duplicates_removed}")


[Stage 9:=======>                                                   (1 + 7) / 8]

Duplicate orders removed: 20025


In [6]:
# ------------------------------------------------------------ 
# 2. Identify and remove bad order amounts 
# ------------------------------------------------------------ 
# A bad amount is defined as: 
    # - NULL / missing 
    # - zero 
    # - negative 
# # These rows are counted before being removed so that the # cleaning process can be fully accounted for.

bad_amount_condition = (
    F.col("amount").isNull() |   # case 1: amount is missing/empty in the CSV
    (F.col("amount") <= 0)       # case 2 and 3: amount is zero or negative
)

# Select bad rows so they can be counted before removal.
bad_amounts = orders_dedup.filter(bad_amount_condition)

# Count how many bad rows there are
bad_amount_count = bad_amounts.count()
print(f"Bad amount rows: {bad_amount_count}")


bad_amounts.show(5)

Bad amount rows: 15022
+--------+-----------+----------+------+-------------------+-------+
|order_id|customer_id|product_id|amount|                 ts|country|
+--------+-----------+----------+------+-------------------+-------+
|    4300|       7867|       359|  null|2024-02-03 19:58:20|     DE|
|   16397|       4806|       333|-194.1|2025-08-16 00:58:33|     NP|
|   22571|       1020|       319|-27.42|2023-10-22 09:54:49|     NP|
|   38633|       3042|       177|-32.13|2024-11-10 07:52:01|     NP|
|   53423|       8781|        26|-67.78|2023-04-22 16:01:11|     GB|
+--------+-----------+----------+------+-------------------+-------+
only showing top 5 rows



In [7]:
# Keep only rows with valid amounts.
# "~" means NOT. This DataFrame is the input for the next step (timestamps).
orders_valid_amount = orders_dedup.filter(~bad_amount_condition)

print(f"Rows after amount cleaning: {orders_valid_amount.count()}")

[Stage 24:=======>                                                  (1 + 7) / 8]

Rows after amount cleaning: 984978


In [8]:
# Quick check: good rows + bad rows must equal the rows we had before this step.
# This should print True.
print("Adds up?", orders_valid_amount.count() + bad_amount_count == orders_dedup.count())

[Stage 36:=======>                                                  (1 + 7) / 8]

Adds up? True


In [9]:
# ------------------------------------------------------------ 
# 3. Parse timestamps and remove invalid timestamps 
# ------------------------------------------------------------ 
# The raw 'ts' column was loaded as STRING because the input 
# contains invalid timestamp values. Convert it to a timestamp 
# using the expected format. Invalid values become NULL.


orders_parsed = orders_valid_amount.withColumn(
    "ts_parsed",
    F.to_timestamp("ts", "yyyy-MM-dd HH:mm:ss")
)

# Count rows that failed timestamp parsing before removing them.
bad_timestamp_count = orders_parsed.filter(
    F.col("ts_parsed").isNull()
).count()

print(f"Invalid timestamp rows: {bad_timestamp_count}")

[Stage 42:=======>                                                  (1 + 7) / 8]

Invalid timestamp rows: 4859


In [10]:
# Keep successfully parsed timestamps, remove the original 
# string column, and rename the parsed column back to 'ts'.

clean_orders = (
    orders_parsed
    .filter(F.col("ts_parsed").isNotNull())
    .drop("ts")
    .withColumnRenamed("ts_parsed", "ts")
)

clean_orders.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- country: string (nullable = true)
 |-- ts: timestamp (nullable = true)



In [11]:
# ------------------------------------------------------------ 
# 4. Deduplicate customers 
# ------------------------------------------------------------ 
# The assignment requires keeping the customer record with 
# the latest signup_date for each customer_id. 
# row_number() assigns rank 1 to the latest signup date.


customer_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(F.col("signup_date").desc())   # latest date first
)

clean_customers = (
    customers
    .withColumn("rn", F.row_number().over(customer_window))  # 1 = latest row per customer
    .filter(F.col("rn") == 1)                                # keep only the latest
    .drop("rn")
)

print("Original customers:", customers.count())
print("Clean customers:", clean_customers.count())

Original customers: 10300
Clean customers: 10000


In [12]:
# ------------------------------------------------------------ 
# 5. Identify orphan orders 
# ------------------------------------------------------------ 
# An orphan order is an order whose customer_id does not exist 
# in the cleaned customer table. 
# left_anti keeps only orders with no matching customer. 
# Orphan orders are kept separately rather than silently deleted.

orphan_orders = clean_orders.join(
    clean_customers.select("customer_id"),
    on="customer_id",
    how="left_anti"      # keeps orders with NO match
)

orphan_count = orphan_orders.count()

print(f"Orphan orders: {orphan_count}")

[Stage 60:=======>                                                  (1 + 7) / 8]

Orphan orders: 9892


In [13]:
# ------------------------------------------------------------ 
# 6. Keep orders belonging to valid customers 
# ------------------------------------------------------------ 
# left_semi keeps only orders that have a matching customer_id. 
# Unlike an inner join, it does not add columns from the # customer DataFrame.

clean_orders = clean_orders.join(
    clean_customers.select("customer_id"),
    on="customer_id",
    how="left_semi"       # keeps orders WITH a match, adds no columns
)

clean_order_count = clean_orders.count()
print(f"Clean orders remaining: {clean_order_count}")

[Stage 69:=======>                                                  (1 + 7) / 8]

Clean orders remaining: 970227


In [77]:
# ---- Save orphans in their own folder (we set them aside, not delete them) ----
OUT = "out"


orphan_orders.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(f"{OUT}/orphan_orders")

In [78]:
# ---- Final report ----

print("\n===== PART A DATA QUALITY REPORT =====")
print(f"Orders started:       {orders_initial:,}")
print(f"Duplicates removed:   {duplicates_removed:,}")
print(f"Bad amounts removed:  {bad_amount_count:,}")
print(f"Bad timestamps:       {bad_timestamp_count:,}")
print(f"Orphans set aside:    {orphan_count:,}")
print(f"Clean orders:         {clean_order_count:,}")
print(f"Customers started:    {customers_initial:,}")
print(f"Clean customers:      {clean_customers.count():,}")

# Proof the accounting is consistent -> output must be true
print("Adds up?",
      orders_initial - duplicates_removed - bad_amount_count
      - bad_timestamp_count - orphan_count == clean_order_count)


===== PART A DATA QUALITY REPORT =====
Orders started:       1,020,025
Duplicates removed:   20,025
Bad amounts removed:  15,022
Bad timestamps:       4,859
Orphans set aside:    9,892
Clean orders:         970,227
Customers started:    10,300
Clean customers:      10,000
Adds up? True


## Part B :- Business Question
#### The following queries answer the six required business 
#### questions using the cleaned orders and reference datasets. 
#### Each result is also exported as a CSV file.

In [91]:
# ------------------------------------------------------------ 
# Q1: Revenue and number of orders per country 
# ------------------------------------------------------------ 
# Country is taken from the orders table. 
# Two implementations are provided: 
    # 1. Spark DataFrame API 
    # 2. Spark SQL using a temporary view 
# Both implementations should produce the same result.

# Version 1: DataFrame API
country_revenue_api = (
    clean_orders
    .groupBy("country")
    .agg(
        F.sum("amount").cast("decimal(18,2)").alias("total_revenue"), 
        F.count("*").alias("number_of_order")
    )
    .orderBy(F.col("total_revenue").desc())    # highest revenue first
)

country_revenue_api.show()

[Stage 760:=======>                                                 (1 + 7) / 8]

+-------+-------------+---------------+
|country|total_revenue|number_of_order|
+-------+-------------+---------------+
|     NP|  47198473.97|         582205|
|     IN|   6293715.51|          77624|
|     US|   6270492.34|          77407|
|     JP|   4677453.51|          57870|
|     AU|   3950063.47|          48831|
|     DE|   3944311.65|          48440|
|     GB|   3159150.30|          39056|
|     BR|   3146709.68|          38794|
+-------+-------------+---------------+



In [20]:
country_revenue_api.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [total_revenue#326 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(total_revenue#326 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=2377]
      +- HashAggregate(keys=[country#5], functions=[sum(amount#3), count(1)])
         +- Exchange hashpartitioning(country#5, 200), ENSURE_REQUIREMENTS, [plan_id=2374]
            +- HashAggregate(keys=[country#5], functions=[partial_sum(amount#3), partial_count(1)])
               +- HashAggregate(keys=[order_id#0, customer_id#1, ts#4, country#5, amount#3, product_id#2], functions=[])
                  +- Exchange hashpartitioning(order_id#0, customer_id#1, ts#4, country#5, amount#3, product_id#2, 200), ENSURE_REQUIREMENTS, [plan_id=2370]
                     +- HashAggregate(keys=[order_id#0, customer_id#1, ts#4, country#5, knownfloatingpointnormalized(normalizenanandzero(amount#3)) AS amount#3, product_id#2], functions=[])
                        +- BroadcastHashJoin

In [92]:
# Create a temporary view so the same analysis can be performed using Spark SQL.

clean_orders.createOrReplaceTempView("clean_orders")

# Version 2: Spark SQL on a temp view
country_revenue_sql = spark.sql("""
    SELECT 
        country,
        CAST(SUM(amount) AS DECIMAL(18,2)) AS total_revenue,
        COUNT(*) AS number_of_orders
    FROM clean_orders
    GROUP BY country
    ORDER BY total_revenue DESC
""")

country_revenue_sql.show()


[Stage 773:=======>                                                 (1 + 7) / 8]

+-------+-------------+----------------+
|country|total_revenue|number_of_orders|
+-------+-------------+----------------+
|     NP|  47198473.97|          582205|
|     IN|   6293715.51|           77624|
|     US|   6270492.34|           77407|
|     JP|   4677453.51|           57870|
|     AU|   3950063.47|           48831|
|     DE|   3944311.65|           48440|
|     GB|   3159150.30|           39056|
|     BR|   3146709.68|           38794|
+-------+-------------+----------------+



In [23]:
country_revenue_sql.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [total_revenue#362 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(total_revenue#362 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=2815]
      +- HashAggregate(keys=[country#5], functions=[sum(amount#3), count(1)])
         +- Exchange hashpartitioning(country#5, 200), ENSURE_REQUIREMENTS, [plan_id=2812]
            +- HashAggregate(keys=[country#5], functions=[partial_sum(amount#3), partial_count(1)])
               +- HashAggregate(keys=[order_id#0, customer_id#1, ts#4, country#5, amount#3, product_id#2], functions=[])
                  +- Exchange hashpartitioning(order_id#0, customer_id#1, ts#4, country#5, amount#3, product_id#2, 200), ENSURE_REQUIREMENTS, [plan_id=2808]
                     +- HashAggregate(keys=[order_id#0, customer_id#1, ts#4, country#5, knownfloatingpointnormalized(normalizenanandzero(amount#3)) AS amount#3, product_id#2], functions=[])
                        +- BroadcastHashJoin

In [24]:
country_revenue_api.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(f"{OUT}/q1_country_revenue")

In [25]:
country_revenue_sql.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(f"{OUT}/q1_country_revenue_sql")

In [98]:
# Verify that the DataFrame API and Spark SQL implementations
# of Q1 produce identical results.

q1_api_check = country_revenue_api.orderBy("country").collect()
q1_sql_check = country_revenue_sql.orderBy("country").collect()

print("Q1 API and SQL results identical:", q1_api_check == q1_sql_check)

[Stage 898:=======>                                                 (1 + 7) / 8]

Q1 API and SQL results identical: True


In [93]:
# ------------------------------------------------------------ 
# Q2: Top 10 customers by lifetime revenue 
# ------------------------------------------------------------ 
# First aggregate orders by customer_id and calculate lifetime revenue. 
# The customer table is joined AFTER aggregation. This reduces 
# the amount of data involved in the join because the join is 
# performed on one row per customer rather than one row per 
# order. 
# Only customer_id and name are selected from customers to 
# avoid unnecessary columns and potential column-name clashes.

top_customers = (
    clean_orders
    .groupBy("customer_id")
    .agg(
        F.sum("amount").cast("decimal(18,2)").alias("lifetime_revenue")
    )
    .join(
        clean_customers.select("customer_id", "name"),
        on="customer_id",
        how="inner"
    )
    .select("customer_id", "name", "lifetime_revenue") 
    .orderBy(F.col("lifetime_revenue").desc())
    .limit(10)
)

top_customers.show()

[Stage 786:=====================>                                   (3 + 5) / 8]

+-----------+--------------+----------------+
|customer_id|          name|lifetime_revenue|
+-----------+--------------+----------------+
|       8059|  Kiran Sharma|        12435.19|
|       1463|    Gita Karki|        11906.92|
|       7706|  Rohan Sharma|        11827.24|
|       4788|  Bibek Gurung|        11810.08|
|       1967|Aarav Adhikari|        11727.24|
|       1366|Suman Shrestha|        11677.78|
|       9662|   Suman Magar|        11661.75|
|       9668|   Sita Sharma|        11641.56|
|       8423|      Priya KC|        11620.78|
|       3237|   Hari Tamang|        11613.93|
+-----------+--------------+----------------+



In [27]:
top_customers.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(f"{OUT}/q2_top_customers")

In [94]:
# ------------------------------------------------------------ 
# Q3: Revenue per product category 
# ------------------------------------------------------------ 
# Join orders with the product reference data to obtain the 
# product category, then aggregate revenue by category. 
# products is tiny (500 rows), so this join is cheap. Selecting only product_id and category keeps unneeded columns out of the join

category_revenue = (
    clean_orders
    .join(
        products.select("product_id", "category"),
        on="product_id",
        how="inner"  # keep only orders whose product exists in the products table
    )
    .groupBy("category")
    .agg(
        F.sum("amount").cast("decimal(18,2)").alias("total_revenue")
    )
    .orderBy(F.col("total_revenue").desc())
)

category_revenue.show()

[Stage 796:=======>                                                 (1 + 7) / 8]

+-----------+-------------+
|   category|total_revenue|
+-----------+-------------+
|      books|  12033285.79|
|       toys|  10553727.95|
|       home|  10520860.11|
|     sports|  10422150.15|
|   clothing|   9719113.21|
|electronics|   8898944.74|
|     beauty|   8833604.60|
|    grocery|   7658683.88|
+-----------+-------------+



In [29]:
category_revenue.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(f"{OUT}/q3_category_revenue")

In [95]:
# ------------------------------------------------------------ 
# Q4: Average order value per signup-year cohort 
# ------------------------------------------------------------ 
# A customer cohort is defined by the year in which the customer signed up. 
# Orders are joined with customer signup dates, the signup year 
# is extracted, and the average order value is calculated for # each cohort.

cohort_average_order_value = (
    clean_orders.join(
        clean_customers.select("customer_id", "signup_date"),  # Only customer_id and signup_date are selected, so "country" cannot clash.
        on="customer_id",
        how="inner"
    )
    .withColumn(
        "signup_year", F.year("signup_date")) # cohort = year the customer signed up
    .groupBy("signup_year")
    .agg(
        F.avg("amount").cast("decimal(18,2)").alias("average_order_value"),
        F.count("*").alias("number_of_orders")
    )
    .orderBy("signup_year")
)
cohort_average_order_value.show()

[Stage 818:=======>                                                 (1 + 7) / 8]

+-----------+-------------------+----------------+
|signup_year|average_order_value|number_of_orders|
+-----------+-------------------+----------------+
|       2020|              81.01|          166183|
|       2021|              81.22|          175180|
|       2022|              81.07|          176625|
|       2023|              81.05|          182197|
|       2024|              81.10|          179838|
|       2025|              80.72|           90204|
+-----------+-------------------+----------------+



In [31]:
(cohort_average_order_value.coalesce(1).write
    .mode("overwrite")
    .option("header", True)
    .csv(f"{OUT}/q4_signup_cohort_avg_order_value"))

In [96]:
# ------------------------------------------------------------ 
# Q5: Monthly revenue trend for 2025 
# ------------------------------------------------------------ 
# Filter orders to 2025, extract the year-month from the 
# timestamp, and calculate total revenue for each month. 
# The yyyy-MM format allows the months to sort chronologically 
# when ordered as strings.

monthly_revenue_2025 = (
    clean_orders
    .filter(F.year("ts") == 2025)     # keep only orders placed in 2025
    .withColumn("month", F.date_format("ts", "yyyy-MM"))   # e.g. "2025-03"; sorts correctly as text
    .groupBy("month")
    .agg(F.sum("amount").cast("decimal(18,2)").alias("monthly_revenue"))
    .orderBy("month")    # January first, December last
)

monthly_revenue_2025.show()

[Stage 831:=======>                                                 (1 + 7) / 8]

+-------+---------------+
|  month|monthly_revenue|
+-------+---------------+
|2025-01|     1889878.75|
|2025-02|     1746124.91|
|2025-03|     1916595.27|
|2025-04|     1844932.60|
|2025-05|     1911622.05|
|2025-06|     1809383.52|
|2025-07|     1900078.67|
|2025-08|     1918071.74|
|2025-09|     1831118.64|
|2025-10|     1910202.06|
|2025-11|     1849856.01|
|2025-12|     1900173.33|
+-------+---------------+



In [33]:
(monthly_revenue_2025.coalesce(1).write 
    .mode("overwrite") 
    .option("header", True) 
    .csv(f"{OUT}/q5_monthly_revenue_2025"))

In [97]:
# ------------------------------------------------------------ 
# Q6: Best-selling category for each country 
# ------------------------------------------------------------ 

# Step 1: Calculate total revenue for every country-category pair.

country_category_revenue = (
    clean_orders
    .join(
        products.select("product_id", "category"),
        on="product_id",
        how="inner"
    )
    .groupBy("country", "category")
    .agg(
        F.sum("amount").cast("decimal(18,2)").alias("category_revenue")
    )
)

# Step 2: Create a window for each country and order categories by 
# revenue in descending order. 
# Category is included as a secondary sort key so that ties 
# are resolved deterministically.

category_window = (
    Window
    .partitionBy("country")
    .orderBy(F.col("category_revenue").desc(),  "category")
)

# Step 3: Assign a row number within each country and keep the category with the highest revenue.

best_category_by_country = (
    country_category_revenue
    .withColumn("rank",F.row_number().over(category_window))
    .filter(F.col("rank") == 1)  
    .select("country", "category", "category_revenue") 
    .orderBy("country")
)



best_category_by_country.show()

[Stage 844:=======>                                                 (1 + 7) / 8]

+-------+--------+----------------+
|country|category|category_revenue|
+-------+--------+----------------+
|     AU|   books|       591300.51|
|     BR|   books|       508307.43|
|     DE|   books|       599115.98|
|     GB|   books|       488772.76|
|     IN|   books|       938226.47|
|     JP|   books|       705472.62|
|     NP|   books|      7243459.58|
|     US|   books|       958630.44|
+-------+--------+----------------+



In [35]:
(best_category_by_country
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(f"{OUT}/q6_best_category_by_country"))

In [99]:
# ============================================================
# C1: Broadcast Join Optimization
# ============================================================
# Recreate the slow configuration left by the previous engineer.
# Broadcast joins and Adaptive Query Execution (AQE) are disabled
# so that the baseline uses a shuffle-based join.
#
# The optimized version below explicitly broadcasts the small
# customer table to avoid the shuffle required by a Sort-Merge Join.
# ============================================================

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
spark.conf.set("spark.sql.adaptive.enabled", "false")

print("Broadcast threshold:",
      spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

print("AQE enabled:",
      spark.conf.get("spark.sql.adaptive.enabled"))

Broadcast threshold: -1
AQE enabled: false


In [100]:
# C1 BEFORE: Baseline without broadcast join
# With automatic broadcasting disabled, Spark should use a
# shuffle-based join such as SortMergeJoin.

c1_before = (
    clean_orders
    .join(
        clean_customers.select("customer_id"),
        on="customer_id",
        how="inner"
    )
    .groupBy(clean_orders["country"])
    .agg(
        F.sum(clean_orders["amount"]).cast("decimal(18,2)").alias("total_revenue")
    )
)

c1_before.show()

c1_before.explain()

[Stage 922:=======>                                                 (1 + 7) / 8]

+-------+-------------+
|country|total_revenue|
+-------+-------------+
|     AU|   3950063.47|
|     IN|   6293715.51|
|     GB|   3159150.30|
|     US|   6270492.34|
|     NP|  47198473.97|
|     JP|   4677453.51|
|     DE|   3944311.65|
|     BR|   3146709.68|
+-------+-------------+

== Physical Plan ==
*(12) HashAggregate(keys=[country#2287], functions=[sum(amount#2285)])
+- Exchange hashpartitioning(country#2287, 16), ENSURE_REQUIREMENTS, [plan_id=27057]
   +- *(11) HashAggregate(keys=[country#2287], functions=[partial_sum(amount#2285)])
      +- *(11) Project [amount#2285, country#2287]
         +- *(11) SortMergeJoin [customer_id#2283], [customer_id#2922], Inner
            :- *(7) SortMergeJoin [customer_id#2283], [customer_id#2294], LeftSemi
            :  :- *(3) Sort [customer_id#2283 ASC NULLS FIRST], false, 0
            :  :  +- Exchange hashpartitioning(customer_id#2283, 16), ENSURE_REQUIREMENTS, [plan_id=26972]
            :  :     +- *(2) HashAggregate(keys=[order_id#

In [101]:
# C1 AFTER: Explicit broadcast join
# clean_customers is small enough to broadcast, allowing Spark
# to avoid the shuffle-based join used in the baseline.
# broadcast() explicitly forces the broadcast strategy even
# though the automatic broadcast threshold is disabled.

from pyspark.sql.functions import broadcast

c1_after = (
    clean_orders
    .join(
        broadcast(clean_customers),
        on="customer_id",
        how="inner"
    )
    .groupBy(clean_orders["country"])
    .agg(
        F.sum(clean_orders["amount"]).cast("decimal(18,2)").alias("total_revenue")
    )
)

c1_after.show()
c1_after.explain()

[Stage 939:=================================================>       (7 + 1) / 8]

+-------+-------------+
|country|total_revenue|
+-------+-------------+
|     AU|   3950063.47|
|     IN|   6293715.51|
|     GB|   3159150.30|
|     US|   6270492.34|
|     NP|  47198473.97|
|     JP|   4677453.51|
|     DE|   3944311.65|
|     BR|   3146709.68|
+-------+-------------+

== Physical Plan ==
*(11) HashAggregate(keys=[country#2287], functions=[sum(amount#2285)])
+- Exchange hashpartitioning(country#2287, 16), ENSURE_REQUIREMENTS, [plan_id=27580]
   +- *(10) HashAggregate(keys=[country#2287], functions=[partial_sum(amount#2285)])
      +- *(10) Project [amount#2285, country#2287]
         +- *(10) BroadcastHashJoin [customer_id#2283], [customer_id#2964], Inner, BuildRight, false
            :- *(10) SortMergeJoin [customer_id#2283], [customer_id#2294], LeftSemi
            :  :- *(3) Sort [customer_id#2283 ASC NULLS FIRST], false, 0
            :  :  +- Exchange hashpartitioning(customer_id#2283, 16), ENSURE_REQUIREMENTS, [plan_id=27482]
            :  :     +- *(2) HashA

In [102]:
# ============================================================
# C2: Shuffle Partition Optimization
# ============================================================
# Compare the default 200 shuffle partitions with 16 partitions.
# The local Spark environment has 8 default parallel tasks, so
# 16 partitions provides approximately two shuffle partitions
# per available task.
# ============================================================

print("Default parallelism:",
      spark.sparkContext.defaultParallelism)

# C2 BEFORE: Use the default 200 shuffle partitions.
spark.conf.set("spark.sql.shuffle.partitions", 200)

print(
    "Shuffle partitions:",
    spark.conf.get("spark.sql.shuffle.partitions")
)


c2_before = (
    clean_orders
    .join(
        broadcast(clean_customers),
        on="customer_id",
        how="inner"
    )
    .groupBy(clean_orders["country"])
    .agg(
        F.sum(clean_orders["amount"]).cast("decimal(18,2)").alias("total_revenue")
    )
)

c2_before.collect()

Default parallelism: 8
Shuffle partitions: 200


[Row(country='NP', total_revenue=Decimal('47198473.97')),
 Row(country='AU', total_revenue=Decimal('3950063.47')),
 Row(country='GB', total_revenue=Decimal('3159150.30')),
 Row(country='BR', total_revenue=Decimal('3146709.68')),
 Row(country='DE', total_revenue=Decimal('3944311.65')),
 Row(country='US', total_revenue=Decimal('6270492.34')),
 Row(country='IN', total_revenue=Decimal('6293715.51')),
 Row(country='JP', total_revenue=Decimal('4677453.51'))]

In [103]:
# C2 AFTER: Reduce shuffle partitions from 200 to 16.
# The query logic remains unchanged so that the comparison
# isolates the effect of shuffle partition count.

spark.conf.set("spark.sql.shuffle.partitions", 16)
spark.sparkContext.setJobDescription("C2 after: 16 shuffle partitions")

c2_after = (
    clean_orders
    .join(
        broadcast(clean_customers),
        on="customer_id",
        how="inner"
    )
    .groupBy(clean_orders["country"])
    .agg(
        F.sum(clean_orders["amount"]).cast("decimal(18,2)").alias("total_revenue")
    )
)

c2_after.collect()

[Row(country='AU', total_revenue=Decimal('3950063.47')),
 Row(country='IN', total_revenue=Decimal('6293715.51')),
 Row(country='GB', total_revenue=Decimal('3159150.30')),
 Row(country='US', total_revenue=Decimal('6270492.34')),
 Row(country='NP', total_revenue=Decimal('47198473.97')),
 Row(country='JP', total_revenue=Decimal('4677453.51')),
 Row(country='DE', total_revenue=Decimal('3944311.65')),
 Row(country='BR', total_revenue=Decimal('3146709.68'))]

In [104]:
spark.sparkContext.setJobDescription(None)

In [105]:
print(
    "Final shuffle partitions:",
    spark.conf.get("spark.sql.shuffle.partitions")
)

Final shuffle partitions: 16


In [106]:
# ============================================================
# C3: Adaptive Query Execution (AQE)
# ============================================================
# Compare the same aggregation with AQE disabled and enabled.
# AQE allows Spark to optimize the physical execution plan at
# runtime based on the actual data statistics.
# ============================================================

# C3 BEFORE: AQE disabled

spark.conf.set("spark.sql.adaptive.enabled", "false")

c3_before = (
    clean_orders
    .groupBy("country")
    .agg(
        F.sum("amount").alias("total_revenue")
    )
)

c3_before.collect()

[Row(country='AU', total_revenue=3950063.4700000007),
 Row(country='IN', total_revenue=6293715.509999996),
 Row(country='GB', total_revenue=3159150.3000000007),
 Row(country='US', total_revenue=6270492.339999996),
 Row(country='NP', total_revenue=47198473.969999954),
 Row(country='JP', total_revenue=4677453.510000002),
 Row(country='DE', total_revenue=3944311.6499999994),
 Row(country='BR', total_revenue=3146709.6799999997)]

In [107]:
# C3 AFTER: AQE enabled
spark.conf.set("spark.sql.adaptive.enabled", "true")

c3_after = (
    clean_orders
    .groupBy("country")
    .agg(
        F.sum("amount").alias("total_revenue")
    )
)

c3_after.collect()

[Row(country='AU', total_revenue=3950063.469999998),
 Row(country='IN', total_revenue=6293715.509999999),
 Row(country='GB', total_revenue=3159150.3000000017),
 Row(country='US', total_revenue=6270492.3400000045),
 Row(country='NP', total_revenue=47198473.970000066),
 Row(country='JP', total_revenue=4677453.509999999),
 Row(country='DE', total_revenue=3944311.649999998),
 Row(country='BR', total_revenue=3146709.6799999983)]

In [108]:
# ============================================================
# C4: Cache a Reused DataFrame
# ============================================================
# Create one filtered and joined DataFrame that will be reused
# by two different aggregations.
#
# Without caching, Spark may recompute the filtering and join
# for each action. Caching stores the computed DataFrame so
# subsequent actions can reuse it.
# ============================================================

c4_base = (
    clean_orders
    .filter(F.col("amount") > 50)
    .join(
        products.select("product_id", "category"),
        on="product_id",
        how="inner"
    )
)

c4_base.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [product_id#2284, customer_id#2283, order_id#2282, amount#2285, country#2287, ts#2451, category#2303]
   +- SortMergeJoin [product_id#2284], [product_id#2302], Inner
      :- Sort [product_id#2284 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(product_id#2284, 16), ENSURE_REQUIREMENTS, [plan_id=28733]
      :     +- Project [customer_id#2283, order_id#2282, product_id#2284, amount#2285, country#2287, ts_parsed#2419 AS ts#2451]
      :        +- SortMergeJoin [customer_id#2283], [customer_id#2294], LeftSemi
      :           :- Sort [customer_id#2283 ASC NULLS FIRST], false, 0
      :           :  +- Exchange hashpartitioning(customer_id#2283, 16), ENSURE_REQUIREMENTS, [plan_id=28727]
      :           :     +- HashAggregate(keys=[order_id#2282, customer_id#2283, ts#2286, country#2287, amount#2285, product_id#2284], functions=[])
      :           :        +- Exchange hashpartitioning(order_id#2282, cus

In [109]:
# C4 BEFORE: Run the first aggregation without caching.
c4_before_1 = (
    c4_base
    .groupBy("category")
    .agg(
        F.sum("amount").alias("total_revenue")
    )
)

c4_before_1.show()

[Stage 986:=======>                                                 (1 + 7) / 8]

+-----------+--------------------+
|   category|       total_revenue|
+-----------+--------------------+
|       toys|   9185546.770000014|
|    grocery|   6651536.320000014|
|       home|   9136563.239999996|
|electronics|    7745466.82000001|
|     beauty|   7690825.179999983|
|     sports|   9065341.100000005|
|   clothing|   8455596.130000014|
|      books|1.0476900920000024E7|
+-----------+--------------------+



In [110]:
# C4 BEFORE: Run a second aggregation.
# Without caching, the common input may need to be recomputed.

c4_before_2 = (
    c4_base
    .groupBy("category")
    .agg(
        F.count("*").alias("number_of_orders")
    )
)

c4_before_2.show()

[Stage 1008:=======>                                                (1 + 7) / 8]

+-----------+----------------+
|   category|number_of_orders|
+-----------+----------------+
|       toys|           70607|
|    grocery|           51227|
|       home|           70231|
|electronics|           59672|
|     sports|           69660|
|     beauty|           59013|
|   clothing|           65236|
|      books|           80423|
+-----------+----------------+



In [111]:
# Cache the reusable DataFrame and materialize the cache.
c4_base_cached = c4_base.cache()
c4_base_cached.count()

526069

In [113]:
# C4 AFTER: Reuse the cached DataFrame for the first aggregation.
c4_after_1 = (
    c4_base_cached
    .groupBy("category")
    .agg(
        F.sum("amount").alias("total_revenue")
    )
)

c4_after_1.show()

+-----------+--------------------+
|   category|       total_revenue|
+-----------+--------------------+
|       toys|   9185546.770000001|
|       home|   9136563.240000004|
|    grocery|          6651536.32|
|electronics|   7745466.819999999|
|     beauty|   7690825.180000002|
|     sports|   9065341.099999998|
|   clothing|   8455596.129999997|
|      books|1.0476900920000006E7|
+-----------+--------------------+



In [114]:
# C4 AFTER: Reuse the same cached DataFrame for the second aggregation.
c4_after_2 = (
    c4_base_cached
    .groupBy("category")
    .agg(
        F.count("*").alias("number_of_orders")
    )
)

c4_after_2.show()

+-----------+----------------+
|   category|number_of_orders|
+-----------+----------------+
|       toys|           70607|
|       home|           70231|
|    grocery|           51227|
|electronics|           59672|
|     beauty|           59013|
|     sports|           69660|
|   clothing|           65236|
|      books|           80423|
+-----------+----------------+



In [115]:
# Remove the cached DataFrame from memory after the experiment.
c4_base_cached.unpersist()

DataFrame[product_id: int, customer_id: int, order_id: int, amount: double, country: string, ts: timestamp, category: string]

In [23]:
# ============================================================
# C5: Replace Python UDF with Native Spark Expressions
# ============================================================
# The original implementation uses a Python UDF to classify
# orders based on their amount.
#
# Rules:
#   amount > 200  -> large
#   amount > 50  -> medium
#   otherwise     -> small
#
# This version is used as the BEFORE performance baseline.
# ============================================================

@F.udf("string")
def size_label(a):
    return "large" if a > 200 else ("medium" if a > 50 else "small")


# Apply the Python UDF to create the size_label column
c5_udf = (
    clean_orders
    .withColumn("size_label", size_label(F.col("amount")))
)

In [24]:
c5_udf.groupBy("size_label").count().show()

[Stage 184:>                                                        (0 + 8) / 8]

+----------+------+
|size_label| count|
+----------+------+
|    medium|445413|
|     small|444158|
|     large| 80656|
+----------+------+



In [25]:
c5_udf.groupBy("size_label").count().show()

[Stage 193:>                                                        (0 + 8) / 8]

+----------+------+
|size_label| count|
+----------+------+
|    medium|445413|
|     small|444158|
|     large| 80656|
+----------+------+



In [26]:
# ============================================================
# C5: Native Spark Expression Version
# ============================================================
# Replace the Python UDF with Spark's built-in when/otherwise
# expressions.
#
# The business logic is intentionally kept identical:
#   amount > 200  -> large
#   amount > 100  -> medium
#   otherwise     -> small
#
# Native Spark expressions avoid the Python UDF execution
# overhead and allow Spark to optimize the expression directly.
# ============================================================

c5_native = (
    clean_orders
    .withColumn(
        "size_label",
        F.when(F.col("amount") > 200, "large")
         .when(F.col("amount") > 50, "medium")
         .otherwise("small")
    )
)

c5_native.groupBy("size_label").count().show()

[Stage 200:==============>                                          (2 + 6) / 8]

+----------+------+
|size_label| count|
+----------+------+
|    medium|445413|
|     small|444158|
|     large| 80656|
+----------+------+



In [27]:
c5_native.groupBy("size_label").count().show()

[Stage 209:==============>                                          (2 + 6) / 8]

+----------+------+
|size_label| count|
+----------+------+
|    medium|445413|
|     small|444158|
|     large| 80656|
+----------+------+



In [28]:
# ============================================================
# Verify that the Python UDF and native Spark implementations
# produce exactly the same results.
# ============================================================

udf_counts = (
    c5_udf
    .groupBy("size_label")
    .count()
    .orderBy("size_label")
)

native_counts = (
    c5_native
    .groupBy("size_label")
    .count()
    .orderBy("size_label")
)

print("Python UDF results:")
udf_counts.show()

print("Native Spark results:")
native_counts.show()

Python UDF results:


+----------+------+
|size_label| count|
+----------+------+
|     large| 80656|
|    medium|445413|
|     small|444158|
+----------+------+

Native Spark results:


[Stage 227:=======>                                                 (1 + 7) / 8]

+----------+------+
|size_label| count|
+----------+------+
|     large| 80656|
|    medium|445413|
|     small|444158|
+----------+------+



In [29]:
# Collect both result sets into dictionaries and compare them.
# If the dictionaries are equal, both implementations produced
# identical counts for every order-size category.

udf_result = {
    row["size_label"]: row["count"]
    for row in udf_counts.collect()
}

native_result = {
    row["size_label"]: row["count"]
    for row in native_counts.collect()
}

print("Counts identical:", udf_result == native_result)

[Stage 252:=======>                                                 (1 + 7) / 8]

Counts identical: True


In [32]:
# ============================================================
# C5 Performance Summary
# ============================================================
# The durations below should be taken from the Spark UI SQL tab.
# Use the SECOND run of each implementation as required by
# the assignment.
# ============================================================

udf_duration = 3.0     
native_duration = 2.0    

print("===== C5 PERFORMANCE COMPARISON =====")
print(f"Python UDF duration:       {udf_duration:.2f} seconds")
print(f"Native expression duration:{native_duration:.2f} seconds")

if native_duration > 0:
    speedup = udf_duration / native_duration
    print(f"Speedup:                   {speedup:.2f}x")

print(f"Counts identical:          {udf_result == native_result}")

===== C5 PERFORMANCE COMPARISON =====
Python UDF duration:       3.00 seconds
Native expression duration:2.00 seconds
Speedup:                   1.50x
Counts identical:          True
